In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
from scipy import stats, signal
from typing import Dict, Any, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. SPEED OF LIGHT MULTIVARIATE ENGINES (Pure NumPy Linear Algebra)
# ==============================================================================

def _fast_adf(x: np.ndarray) -> float:
    """Fast Augmented Dickey-Fuller test (1 lag, constant) for stationarity."""
    diff_x = np.diff(x)
    x_lag = x[:-1]
    X = np.vstack([x_lag, np.ones(len(x_lag))]).T
    beta, _, _, _ = np.linalg.lstsq(X, diff_x, rcond=None)
    resid = diff_x - X @ beta
    n = len(diff_x)
    sigma2 = np.sum(resid**2) / (n - 2)
    var_beta = sigma2 * np.linalg.inv(X.T @ X + 1e-12 * np.eye(2))[0, 0]
    return float(beta[0] / np.sqrt(var_beta + 1e-12))

def _fit_var_ols(Y: np.ndarray, lags: int, ridge: float = 1e-5) -> Tuple[List[np.ndarray], np.ndarray, np.ndarray]:
    """Fits VAR(p) using Ridge-regularized OLS for numerical stability."""
    T, N = Y.shape
    if T <= lags * N + 1:
        lags = max(1, (T - 2) // (N + 1))
        
    Y_target = Y[lags:]
    X_parts = [np.ones((T - lags, 1))]
    for i in range(1, lags + 1):
        X_parts.append(Y[lags - i : T - i])
    X = np.hstack(X_parts)
    
    # Ridge Regression: beta = (X^T X + lambda I)^-1 X^T Y
    XtX = X.T @ X + ridge * np.eye(X.shape[1])
    XtY = X.T @ Y_target
    beta = np.linalg.solve(XtX, XtY)
    
    residuals = Y_target - X @ beta
    df = max(1, T - lags - N * lags - 1)
    Sigma = (residuals.T @ residuals) / df
    
    A_matrices = []
    for i in range(lags):
        start_idx = 1 + i * N
        end_idx = 1 + (i + 1) * N
        A_matrices.append(beta[start_idx:end_idx, :].T) # N x N matrices
        
    return A_matrices, Sigma, residuals

def _compute_fevd(A_matrices: List[np.ndarray], Sigma: np.ndarray, steps: int = 10) -> np.ndarray:
    """Forecast Error Variance Decomposition (Orthogonalized via Cholesky)."""
    N = Sigma.shape[0]
    Phis = [np.eye(N)]
    for h in range(1, steps):
        Phi_h = np.zeros((N, N))
        for j in range(1, min(h, len(A_matrices)) + 1):
            Phi_h += A_matrices[j-1] @ Phis[h-j]
        Phis.append(Phi_h)
        
    try:
        P = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        P = np.linalg.cholesky(Sigma + 1e-5 * np.eye(N)) # Fallback for collinearity
        
    Psis = [Phi @ P for Phi in Phis]
    omega = np.zeros((N, N))
    for i in range(steps):
        omega += Psis[i] ** 2 # Element-wise square of impulse responses
        
    row_sums = omega.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return omega / row_sums # fevd[target, source]

def _pairwise_granger_matrix(data: np.ndarray, max_lag: int) -> np.ndarray:
    """Computes NxN Granger Causality p-value matrix."""
    N = data.shape[1]
    p_val_mat = np.ones((N, N))
    T = data.shape[0]
    lags_to_test = sorted(list(set([1, 2, 3, 5, max_lag])))
    lags_to_test = [l for l in lags_to_test if l < T // 3]
    if not lags_to_test: lags_to_test = [1]
    
    for i in range(N):
        for j in range(N):
            if i == j: continue
            x, y = data[:, i], data[:, j] # X -> Y
            best_p = 1.0
            for lag in lags_to_test:
                Y_t = y[lag:]
                X_R = np.ones((T - lag, 1 + lag))
                for k in range(1, lag + 1): X_R[:, k] = y[lag - k : T - k]
                X_U = np.ones((T - lag, 1 + 2 * lag))
                X_U[:, 1:lag+1] = X_R[:, 1:]
                for k in range(1, lag + 1): X_U[:, lag + k] = x[lag - k : T - k]
                
                beta_R, _, _, _ = np.linalg.lstsq(X_R, Y_t, rcond=None)
                beta_U, _, _, _ = np.linalg.lstsq(X_U, Y_t, rcond=None)
                rss_R = np.sum((Y_t - X_R @ beta_R)**2)
                rss_U = np.sum((Y_t - X_U @ beta_U)**2)
                
                df_num, df_denom = lag, T - 2 * lag - 1
                if df_denom > 0 and rss_U > 1e-12:
                    f_stat = ((rss_R - rss_U) / df_num) / (rss_U / df_denom)
                    p_val = stats.f.sf(f_stat, df_num, df_denom)
                    if p_val < best_p: best_p = p_val
            p_val_mat[i, j] = best_p
    return p_val_mat

# ==============================================================================
# 2. MAIN MULTIVARIATE DISPATCHER
# ==============================================================================

def multivariate_timeseries_analysis(df: pl.DataFrame, 
                                     time_col: str | None = None, 
                                     max_lag: int = 10, 
                                     top_k_vars: int = 12,
                                     anomaly_sigma: float = 3.0,
                                     name: str = "north") -> Dict[str, Any]:
    """
    OP Multivariate Time Series Engine.
    Computes VAR, FEVD, Causal Networks, and Systemic Anomalies.
    """
    # 1. Data Prep & Feature Selection (Select Top K by Variance)
    num_cols = [c for c in df.columns if df[c].dtype.is_numeric() and c != time_col]
    if not num_cols: raise ValueError("No numeric columns found.")
    
    if len(num_cols) > top_k_vars:
        variances = [df[c].var() or 0.0 for c in num_cols]
        num_cols = [c for _, c in sorted(zip(variances, num_cols), reverse=True)][:top_k_vars]
        
    if time_col and time_col in df.columns:
        df = df.sort(time_col)
        time_steps = df[time_col].to_numpy()
    else:
        time_steps = np.arange(len(df))
        
    df_clean = df.select(num_cols).drop_nulls()
    data_orig = df_clean.to_numpy().astype(float)
    T, N = data_orig.shape
    
    if T < 30: raise ValueError("Dataset too small for multivariate VAR analysis.")
    
    # 2. Stationarity Check & Auto-Differencing
    adf_stats = [_fast_adf(data_orig[:, i]) for i in range(N)]
    is_stationary = all(stat < -2.86 for stat in adf_stats) # ~5% MacKinnon CV
    data = np.diff(data_orig, axis=0) if not is_stationary else data_orig
    if not is_stationary: 
        T -= 1
        time_steps = time_steps[1:]
        
    var_names = num_cols
    max_lag = min(max_lag, T // 5)
    
    # 3. Core Computations
    A_matrices, Sigma, residuals = _fit_var_ols(data, max_lag)
    fevd_mat = _compute_fevd(A_matrices, Sigma, steps=max_lag)
    p_val_mat = _pairwise_granger_matrix(data, max_lag)
    
    # 4. Systemic Anomaly Detection (Mahalanobis Distance in Residual Space)
    try:
        inv_cov = np.linalg.inv(np.cov(residuals, rowvar=False) + 1e-5 * np.eye(N))
        diff_res = residuals - residuals.mean(axis=0)
        mahal_scores = np.sqrt(np.sum((diff_res @ inv_cov) * diff_res, axis=1))
    except:
        mahal_scores = np.linalg.norm(residuals, axis=1)
        
    mu_m, sigma_m = np.mean(mahal_scores), np.std(mahal_scores)
    anomalies = mahal_scores > (mu_m + anomaly_sigma * sigma_m)
    
    # 5. Graph Theory (Causal Drivers)
    G = nx.DiGraph()
    for i in range(N):
        for j in range(N):
            if i != j and p_val_mat[i, j] < 0.05:
                weight = -np.log10(p_val_mat[i, j] + 1e-12)
                G.add_edge(var_names[i], var_names[j], weight=weight)
                
    drivers = []
    if len(G.edges) > 0:
        pr = nx.pagerank(G, weight='weight')
        in_deg = dict(G.in_degree(weight='weight'))
        out_deg = dict(G.out_degree(weight='weight'))
        for node in G.nodes():
            drivers.append({"var": node, "out": out_deg.get(node, 0), "in": in_deg.get(node, 0), "pr": pr.get(node, 0)})
        drivers.sort(key=lambda x: x["pr"], reverse=True)

    # ==========================================================================
    # 6. BEAUTIFUL VISUALIZATION DASHBOARD
    # ==========================================================================
    fig = plt.figure(figsize=(24, 20))
    fig.patch.set_facecolor('#F8F9FA')
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)
    
    # A. Causal Network (Top Left, spanning 2 cols)
    ax_net = fig.add_subplot(gs[0, 0:2])
    G_vis = nx.DiGraph()
    for i in range(N):
        for j in range(N):
            if i != j and p_val_mat[i, j] < 0.05:
                G_vis.add_edge(var_names[i], var_names[j], weight=-np.log10(p_val_mat[i, j] + 1e-12))
                
    if len(G_vis.edges) > 0:
        pos = nx.circular_layout(G_vis)
        edges = G_vis.edges()
        weights = [G_vis[u][v]['weight'] for u, v in edges]
        max_w = max(weights) if weights else 1
        widths = [2.5 * (w / max_w) + 0.5 for w in weights]
        
        nx.draw_networkx_nodes(G_vis, pos, ax=ax_net, node_color='#0077B6', node_size=1200, alpha=0.9)
        nx.draw_networkx_labels(G_vis, pos, ax=ax_net, font_color='white', font_weight='bold', font_size=11)
        nx.draw_networkx_edges(G_vis, pos, edgelist=edges, width=widths, edge_color='#E63946', 
                               ax=ax_net, connectionstyle="arc3,rad=0.15", arrowsize=25, alpha=0.7)
    else:
        ax_net.text(0.5, 0.5, "No Significant Causal Links (p < 0.05)", ha='center', va='center', fontsize=16)
    ax_net.set_title("Granger Causal Network (Edge Width = Significance)", fontweight='bold', color='#1D3557', fontsize=16)
    ax_net.axis('off')
    
    # B. System State Space (Global PCA) (Top Right)
    ax_state = fig.add_subplot(gs[0, 2])
    U, S, Vt = np.linalg.svd(data - data.mean(axis=0), full_matrices=False)
    proj = data @ Vt.T
    scatter = ax_state.scatter(proj[:, 0], proj[:, 1], c=np.arange(T), cmap='viridis', alpha=0.6, s=20)
    ax_state.set_title("System State Space (PCA Projection)", fontweight='bold', color='#1D3557', fontsize=16)
    ax_state.set_xlabel("PC 1 (Global Trend)")
    ax_state.set_ylabel("PC 2 (Orthogonal Shocks)")
    plt.colorbar(scatter, ax=ax_state, label='Time Progression')
    
    # C. FEVD Heatmap (Middle Left)
    ax_fevd = fig.add_subplot(gs[1, 0])
    sns.heatmap(fevd_mat, annot=True, fmt=".2f", cmap="mako", ax=ax_fevd, 
                xticklabels=var_names, yticklabels=var_names, cbar_kws={'label': 'Variance Explained'})
    ax_fevd.set_title("Forecast Error Variance Decomposition\n(Row=Target, Col=Driver)", fontweight='bold', color='#1D3557', fontsize=14)
    ax_fevd.tick_params(axis='x', rotation=45)
    
    # D. Granger P-Value Matrix (Middle Center)
    ax_granger = fig.add_subplot(gs[1, 1])
    sns.heatmap(-np.log10(p_val_mat + 1e-12), annot=True, fmt=".1f", cmap="inferno", ax=ax_granger, 
                xticklabels=var_names, yticklabels=var_names, cbar_kws={'label': '-log10(p-value)'})
    ax_granger.set_title("Granger Causality Significance\n(Row=Cause, Col=Effect)", fontweight='bold', color='#1D3557', fontsize=14)
    ax_granger.tick_params(axis='x', rotation=45)
    
    # E. Cross-Correlation Matrix (Max Absolute) (Middle Right)
    ax_ccf = fig.add_subplot(gs[1, 2])
    data_std = (data - data.mean(axis=0)) / (data.std(axis=0) + 1e-12)
    max_ccf_mat = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            ccf = signal.correlate(data_std[:, i], data_std[:, j], mode='full', method='fft') / T
            lags = np.arange(-T + 1, T)
            valid = ccf[(lags >= -max_lag) & (lags <= max_lag)]
            max_ccf_mat[i, j] = np.max(np.abs(valid)) if len(valid) > 0 else 0
    sns.heatmap(max_ccf_mat, annot=True, fmt=".2f", cmap="crest", ax=ax_ccf, 
                xticklabels=var_names, yticklabels=var_names, vmin=0, vmax=1)
    ax_ccf.set_title("Max Absolute Cross-Correlation\n(Within ± Lag)", fontweight='bold', color='#1D3557', fontsize=14)
    ax_ccf.tick_params(axis='x', rotation=45)
    
    # F. Systemic Anomaly Detection (Bottom, spanning all 3 cols)
    ax_anom = fig.add_subplot(gs[2, :])
    # Align time steps for anomalies (residuals start after max_lag)
    anom_time = time_steps[max_lag:]
    ax_anom.plot(anom_time, mahal_scores, color='#1D3557', alpha=0.8, label='Systemic Mahalanobis Distance')
    threshold = mu_m + anomaly_sigma * sigma_m
    ax_anom.axhline(threshold, color='#E63946', linestyle='--', linewidth=2.5, label=f'Anomaly Threshold ({anomaly_sigma}σ)')
    ax_anom.fill_between(anom_time, mahal_scores, threshold, 
                         where=(mahal_scores > threshold), color='#E63946', alpha=0.3, interpolate=True)
    ax_anom.set_title("Systemic Anomaly Detection (VAR Residual Space)", fontweight='bold', color='#1D3557', fontsize=16)
    ax_anom.legend(fontsize=12)
    
    # Global Aesthetics
    for ax in fig.axes:
        if ax != ax_net:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#CCCCCC')
            ax.spines['bottom'].set_color('#CCCCCC')
            ax.tick_params(colors='#555555', labelsize=10)
            
    plt.show()

    np.savetxt(f"{name}_fevd_matrix_full.csv", fevd_mat, delimiter=",")
    np.savetxt(f"{name}_granger_p_values_full.csv", p_val_mat, delimiter=",")
    np.savetxt(f"{name}_var_residuals_full.csv", residuals, delimiter=",")
    np.savetxt(f"{name}_anomaly_scores_full.csv", mahal_scores, delimiter=",")

    import pandas as pd
    if drivers:
        pd.DataFrame(drivers).to_csv(f"{name}_causal_drivers.csv", index=False)
    
    # ==========================================================================
    # 7. BEAUTIFUL CONSOLE OUTPUT
    # ==========================================================================
    print(f"\n{'='*75}")
    print(f"  MULTIVARIATE TIME SERIES ANALYSIS: SYSTEM DYNAMICS & CAUSALITY")
    print(f"{'='*75}")
    print(f"  Variables Analyzed: {N:>10}  |  Time Steps: {T:>10}  |  Max Lag: {max_lag:>10}")
    print(f"  System Stability:   {'STATIONARY' if is_stationary else 'DIFFERENCED':>10}  |  Anomalies Found: {np.sum(mahal_scores > threshold):>10}")
    print(f"{'-'*75}")
    print(f"  TOP CAUSAL DRIVERS (PageRank in Granger Network):")
    if drivers:
        for i, drv in enumerate(drivers[:5]):
            print(f"    {i+1}. {drv['var']:<15} | Out-Flow: {drv['out']:<8.2f} | In-Flow: {drv['in']:<8.2f} | PR: {drv['pr']:.4f}")
    else:
        print("    [!] No significant causal links detected at p < 0.05.")
    print(f"{'='*75}\n")
    
    return {
        "fevd_matrix": fevd_mat,
        "granger_p_values": p_val_mat,
        "var_residuals": residuals,
        "anomaly_scores": mahal_scores,
        "drivers": drivers,
        "is_stationary": is_stationary
    }

### Load Dataset

In [ ]:
north_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/north_raw.parquet")
south_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/south_raw.parquet")

### Drop Noise Features

In [ ]:
noise_features = [
    "consecutive_dry_days",
    "total_suspended_solids_mg_L",
    "cumulative_heat_index",
    "drought_x_heat_stress",
    "holiday_weekend_flag",
    "is_weekend"
]

north_df = north_df.drop(noise_features)
south_df = south_df.drop(noise_features)

assert not any(col in north_df.columns for col in noise_features), "North DF still has noise features!"
assert not any(col in south_df.columns for col in noise_features), "South DF still has noise features!"

### Multivariate Time Series Analysis

In [ ]:
multivariate_timeseries_analysis(df=north_df, max_lag=7, top_k_vars=17, anomaly_sigma=3.0, name="north")

In [ ]:
multivariate_timeseries_analysis(df=south_df, max_lag=7, top_k_vars=17, anomaly_sigma=3.0, name="south")

### Hunt Data Leakage

In [ ]:
import polars as pl
import numpy as np
from scipy import stats
import re
import warnings
from typing import Dict, Tuple, Any
from datetime import date, timedelta
from decimal import Decimal

warnings.filterwarnings('ignore')

def to_float(v: Any) -> float:
    """Safely convert any value to a float. Returns 0.0 for unconvertible types."""
    if v is None:
        return 0.0
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, Decimal):
        return float(v)
    if isinstance(v, timedelta):
        return v.total_seconds()
    if isinstance(v, date):
        return float(v.toordinal())
    if isinstance(v, (np.floating, np.integer)):
        return float(v)
    return 0.0

def hunt_data_leakage(df: pl.DataFrame, target: str) -> dict:
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found.")
    
    features = [c for c in df.columns if c != target]
    if not features:
        return {"high_correlation_features": [], "null_dependence_alerts": [], 
                "perfect_or_high_proxy_features": [], "suspicious_id_columns": []}
    
    n_rows = len(df)
    if n_rows < 10:
        raise ValueError("Dataset too small for statistical validity.")
        
    # Convert target to numeric if possible
    target_dtype = df[target].dtype
    if target_dtype.is_numeric():
        target_col = df[target].cast(pl.Float64)
        target_is_num = True
    elif target_dtype == pl.Duration:
        target_col = df[target].dt.total_seconds()
        target_is_num = True
    elif target_dtype in (pl.Decimal,):
        target_col = df[target].cast(pl.Float64)
        target_is_num = True
    else:
        target_col = df[target]
        target_is_num = False
        
    target_rank = target_col.rank(method="average") if target_is_num else None
    target_n_unique = int(target_col.n_unique())
    
    results = {
        "high_correlation_features": [],
        "null_dependence_alerts": [],
        "perfect_or_high_proxy_features": [],
        "suspicious_id_columns": []
    }
    
    def is_effective_numeric(col: str) -> bool:
        dtype = df[col].dtype
        return dtype.is_numeric() or dtype == pl.Duration or dtype in (pl.Decimal,)
        
    def get_numeric_col(col: str) -> pl.Expr:
        dtype = df[col].dtype
        if dtype == pl.Duration:
            return pl.col(col).dt.total_seconds()
        elif dtype in (pl.Decimal,):
            return pl.col(col).cast(pl.Float64)
        else:
            return pl.col(col).cast(pl.Float64)
            
    num_feats = [f for f in features if is_effective_numeric(f)]
    cat_feats = [f for f in features if not is_effective_numeric(f)]
    
    assoc_scores: Dict[str, Tuple[float, str]] = {}
    
    # ---- Association scores ----
    if target_is_num and num_feats:
        target_rank_expr = target_rank if target_rank is not None else pl.col(target)
        exprs = [pl.corr(target_rank_expr, get_numeric_col(f).rank(method="average")).alias(f) for f in num_feats]
        spearman_vals = df.select(exprs).row(0)
        spearman_vals_list: list[float] = [to_float(v) for v in spearman_vals]
        for f, c in zip(num_feats, spearman_vals_list):
            assoc_scores[f] = (abs(c), "spearman")
            
    if not target_is_num and cat_feats:
        target_counts = df.group_by(target).len()
        t_counts = target_counts["len"].to_numpy()
        t_probs = t_counts / n_rows
        H_t = -np.sum(t_probs * np.log(t_probs + 1e-12))
        for f in cat_feats:
            joint = df.group_by([target, f]).len()
            f_counts = df.group_by(f).len()
            joint = joint.join(f_counts, on=f, suffix="_f").join(target_counts, on=target, suffix="_t")
            p_xy = joint["len"].to_numpy() / n_rows
            p_x = joint["len_t"].to_numpy() / n_rows
            p_y = joint["len_f"].to_numpy() / n_rows
            mask = p_xy > 0
            mi = np.sum(p_xy[mask] * np.log(p_xy[mask] / (p_x[mask] * p_y[mask])))
            f_probs = f_counts["len"].to_numpy() / n_rows
            H_f = -np.sum(f_probs * np.log(f_probs + 1e-12))
            nmi = mi / ((H_t + H_f) / 2.0 + 1e-12)
            assoc_scores[f] = (to_float(np.clip(nmi, 0, 1)), "nmi")
            
    if target_is_num and cat_feats:
        grand_mean = to_float(target_col.mean())
        variance = target_col.var()
        ss_total = to_float(variance) * (n_rows - 1) if variance is not None else 0.0
        for f in cat_feats:
            if ss_total > 0.0:
                grouped = df.group_by(f).agg(pl.col(target).mean().alias("mean"), pl.len().alias("count")).drop_nulls()
                mean_vals = grouped["mean"].cast(pl.Float64) if grouped["mean"].dtype != pl.Float64 else grouped["mean"]
                ss_between = (grouped["count"] * (mean_vals - grand_mean)**2).sum()
                eta = np.sqrt(np.clip(to_float(ss_between) / ss_total, 0, 1))
                assoc_scores[f] = (to_float(eta), "eta")
            else:
                assoc_scores[f] = (0.0, "eta")
                
    if not target_is_num and num_feats:
        for f in num_feats:
            temp_df = df.with_columns(get_numeric_col(f).alias("_num_"))
            grand_mean = to_float(temp_df["_num_"].mean())
            variance = temp_df["_num_"].var()
            ss_total = to_float(variance) * (n_rows - 1) if variance is not None else 0.0
            if ss_total > 0.0:
                grouped = temp_df.group_by(target).agg(pl.col("_num_").mean().alias("mean"), pl.len().alias("count")).drop_nulls()
                ss_between = (grouped["count"] * (grouped["mean"] - grand_mean)**2).sum()
                eta = np.sqrt(np.clip(to_float(ss_between) / ss_total, 0, 1))
                assoc_scores[f] = (to_float(eta), "eta")
            else:
                assoc_scores[f] = (0.0, "eta")
                
    for f, (score, method) in assoc_scores.items():
        score_f = to_float(score)
        if score_f > 0.75:
            results["high_correlation_features"].append({
                "feature": f, "association_score": round(score_f, 4), "method": method
            })
        if score_f > 0.98:
            results["perfect_or_high_proxy_features"].append({
                "feature": f, "reason": f"extreme_association ({method}: {score_f:.4f})", "association_score": round(score_f, 4)
            })
            
    # ---- Deterministic proxy ----
    for f in features:
        if any(p["feature"] == f for p in results["perfect_or_high_proxy_features"]):
            continue
        n_f = int(df[f].n_unique())
        if n_f == n_rows:
            continue
        n_pairs = int(df.select([pl.col(f), pl.col(target)]).n_unique())
        assoc_score = to_float(assoc_scores.get(f, (0.0, ""))[0])
        if n_pairs == n_f:
            results["perfect_or_high_proxy_features"].append({
                "feature": f, "reason": "feature_perfectly_predicts_target", "association_score": assoc_score
            })
        elif n_pairs == target_n_unique:
            results["perfect_or_high_proxy_features"].append({
                "feature": f, "reason": "target_perfectly_predicts_feature", "association_score": assoc_score
            })
            
    # ---- Null dependence ----
    null_exprs = [pl.col(f).is_null().cast(pl.Int8).alias(f) for f in features]
    null_df = df.select(null_exprs)
    null_vals = null_df.select([pl.col(f).mean().alias(f) for f in features]).row(0)
    null_rates: list[float] = [to_float(v) for v in null_vals]
    
    if target_is_num:
        target_float = target_col.cast(pl.Float64) if target_col.dtype != pl.Float64 else target_col
        
        # FIX 1: Compute correlation using the null indicator (.is_null()) rather than raw column values
        pb_exprs = [pl.corr(target_float, pl.col(f).is_null().cast(pl.Float64)).alias(f) for f in features]
        
        pb_vals = df.select(pb_exprs).row(0)
        pb_corrs: list[float] = [to_float(v) for v in pb_vals]
        
        for f, corr, null_rate in zip(features, pb_corrs, null_rates):
            nr = to_float(null_rate)
            cr_val = to_float(corr)
            if nr < 0.01 or nr > 0.99 or abs(cr_val) < 1e-12:
                continue
            t_stat = cr_val * np.sqrt((n_rows - 2) / (1 - cr_val**2 + 1e-12))
            p_val = to_float(2 * stats.t.sf(abs(t_stat), n_rows - 2))
            if p_val < 0.05 and abs(cr_val) > 0.15:
                results["null_dependence_alerts"].append({
                    "feature": f, "overall_null_rate": round(nr, 4),
                    "details": {"method": "point_biserial", "correlation": round(abs(cr_val), 4), "p_value": p_val}
                })
    else:
        cats = target_col.unique().to_list()
        cat_to_idx = {c: i for i, c in enumerate(cats)}
        for f, null_rate in zip(features, null_rates):
            nr = to_float(null_rate)
            if nr < 0.01 or nr > 0.99:
                continue
            ct = df.group_by([pl.col(f).is_null().alias("is_null"), target]).len()
            matrix = np.zeros((2, len(cats)))
            for row in ct.iter_rows(named=True):
                r_idx = 1 if row["is_null"] else 0
                c_idx = cat_to_idx[row[target]]
                matrix[r_idx, c_idx] = row["len"]
            if matrix.shape[1] > 1 and matrix.sum(axis=1).min() > 0:
                chi2, p, dof, expected = stats.chi2_contingency(matrix)
                p_chi = to_float(p)
                cat_counts = matrix.sum(axis=0)
                null_rates_per_cat = matrix[1, :] / cat_counts
                max_diff = to_float(null_rates_per_cat.max() - null_rates_per_cat.min())
                if p_chi < 0.05 and max_diff > 0.1:
                    results["null_dependence_alerts"].append({
                        "feature": f, "overall_null_rate": round(nr, 4),
                        "details": {"method": "chi2_null", "p_value": p_chi, "max_null_rate_diff": round(max_diff, 4)}
                    })
                    
    # ---- Suspicious ID columns ----
    id_regex = re.compile(r'id$|_id|^id_|uuid|key|code|number|no$|index|phone|email|zip|postal|ssn|name|user|account|customer|token|hash', re.IGNORECASE)
    name_matches = {f: bool(id_regex.search(f)) for f in df.columns}
    
    card_exprs = [(pl.col(f).drop_nulls().n_unique() / pl.col(f).count()).alias(f) for f in df.columns]
    card_vals = df.select(card_exprs).row(0)
    card_ratios: list[float] = [to_float(v) for v in card_vals]
    
    num_cols = [f for f in df.columns if is_effective_numeric(f)]
    mono_dict: Dict[str, bool] = {}
    
    # FIX 2: Polars `Expr` doesn't have an `.is_sorted()` method in all versions. 
    # Evaluate to a `Series` first, which reliably supports `.is_sorted()`.
    for f in num_cols:
        try:
            s = df.select(get_numeric_col(f).alias(f))[f].drop_nulls()
            if len(s) > 1:
                mono_dict[f] = bool(s.is_sorted(descending=False) or s.is_sorted(descending=True))
            else:
                mono_dict[f] = False
        except Exception:
            mono_dict[f] = False
        
    str_cols = [f for f in df.columns if df[f].dtype in (pl.String, pl.Categorical) or str(df[f].dtype) == "Utf8"]
    str_pattern_dict: Dict[str, bool] = {}
    if str_cols:
        str_n_vals = df.select([pl.col(f).drop_nulls().cast(pl.String).str.len_chars().n_unique().alias(f) for f in str_cols]).row(0)
        str_n_uniq = [int(v) if v is not None else 0 for v in str_n_vals]
        str_max_vals = df.select([pl.col(f).drop_nulls().cast(pl.String).str.len_chars().max().alias(f) for f in str_cols]).row(0)
        str_max_vals = [int(v) if v is not None else 0 for v in str_max_vals]
        for f, n_u, max_l in zip(str_cols, str_n_uniq, str_max_vals):
            str_pattern_dict[f] = (n_u == 1 and max_l > 5)
            
    for col in df.columns:
        valid_count = int(df[col].count())
        if valid_count == 0:
            continue
        
        cr = to_float(card_ratios[df.columns.index(col)])
        is_name = name_matches.get(col, False)
        is_mono = mono_dict.get(col, False)
        is_str_pat = str_pattern_dict.get(col, False)
        
        reasons = []
        if is_name: reasons.append("name_pattern")
        if cr > 0.95: reasons.append("high_cardinality")
        if cr == 1.0: reasons.append("all_unique")
        if is_mono: reasons.append("monotonic_sequence")
        if is_str_pat: reasons.append("fixed_length_string")
        
        suspicious = False
        if is_name and (cr > 0.8 or is_mono):
            suspicious = True
        elif cr > 0.98 and (is_mono or cr == 1.0):
            suspicious = True
        elif is_str_pat and cr > 0.9:
            suspicious = True
            
        if suspicious:
            results["suspicious_id_columns"].append({
                "column": col, "reasons": reasons,
                "cardinality_ratio": round(cr, 4), "is_monotonic": is_mono
            })
            
    return results

In [ ]:
print(hunt_data_leakage(df=north_df, target="water_quality_index"))

In [ ]:
print(hunt_data_leakage(df=south_df, target="water_quality_index"))